# 🚀 Redes recurrentes con Keras: RNN, LSTM y GRU
*Curso de Machine Learning · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** clasificar críticas de cine como positivas o negativas con redes que **leen en orden**, comparar sus
cuatro variantes en la misma tarea y ver por qué la GPU las acelera.

**Datos:** IMDB de Keras: 50.000 críticas en inglés, ya convertidas en números (cada palabra es su posición en un
ranking de frecuencia).

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import tensorflow as tf
import keras
GPUS = tf.config.list_physical_devices("GPU")
print("Colab:", EN_COLAB, "| TensorFlow", tf.__version__, "| GPU:", GPUS[0].name if GPUS else "ninguna")
if not GPUS:
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")
keras.utils.set_random_seed(0)

In [ ]:
VOCABULARIO, LARGO = 10000, 200
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCABULARIO)
if PRUEBA_RAPIDA:
    x_train, y_train, x_test, y_test = x_train[:1500], y_train[:1500], x_test[:500], y_test[:500]
x_train = keras.utils.pad_sequences(x_train, maxlen=LARGO)   # todas de 200 palabras (se corta o se rellena)
x_test = keras.utils.pad_sequences(x_test, maxlen=LARGO)

indice = {i + 3: p for p, i in keras.datasets.imdb.get_word_index().items()}
print(" ".join(indice.get(i, "?") for i in x_train[0] if i > 2)[:300], "→", "positiva" if y_train[0] else "negativa")

## 1. Cuatro redes, la misma tarea

| Capa | Idea |
|---|---|
| `SimpleRNN` | Una memoria que se actualiza palabra a palabra; olvida rápido |
| `LSTM` | Compuertas que deciden qué recordar y qué olvidar: memoria larga |
| `GRU` | Como LSTM, con menos compuertas: más liviana |
| `Bidirectional(LSTM)` | Lee la crítica hacia adelante **y** hacia atrás |

Todas parten de una capa `Embedding`, que convierte cada número de palabra en un vector que se aprende.

In [ ]:
import time
EPOCAS = 1 if PRUEBA_RAPIDA else 3

def red(capa):
    modelo = keras.Sequential([keras.Input((LARGO,)), keras.layers.Embedding(VOCABULARIO, 64), capa,
                               keras.layers.Dense(1, activation="sigmoid")])
    modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return modelo

capas = {
    "SimpleRNN": lambda: keras.layers.SimpleRNN(64),
    "LSTM": lambda: keras.layers.LSTM(64),
    "GRU": lambda: keras.layers.GRU(64),
    "LSTM bidireccional": lambda: keras.layers.Bidirectional(keras.layers.LSTM(64)),
}
resultados = {}
for nombre, capa in capas.items():
    modelo = red(capa())
    inicio = time.perf_counter()
    modelo.fit(x_train, y_train, epochs=EPOCAS, batch_size=128, validation_split=0.1, verbose=0)
    resultados[nombre] = {"exactitud": modelo.evaluate(x_test, y_test, verbose=0)[1],
                          "segundos": time.perf_counter() - inicio, "parámetros": modelo.count_params()}
    print(f"{nombre:20} {resultados[nombre]['exactitud']:.2%} en {resultados[nombre]['segundos']:.0f} s")

In [ ]:
import pandas as pd
pd.DataFrame(resultados).T.round(3)

> **¿Por qué la LSTM vuela en GPU?** Con sus opciones por defecto (activación `tanh`, sin *dropout* recurrente…),
> Keras usa la implementación **cuDNN** de NVIDIA, optimizada para GPU. Si cambias esas opciones, vuelve a la versión
> genérica y se nota en el tiempo. `SimpleRNN` no tiene versión cuDNN.

## Pruébalo tú

Entrena una `GRU` con `recurrent_dropout=0.2` (deja de usar cuDNN) y guarda en `segundos_sin_cudnn` lo que tarda.
Compáralo con la `GRU` normal de la tabla.

In [ ]:
segundos_sin_cudnn = None   # ← tu código

In [ ]:
# Comprobación
if segundos_sin_cudnn is None:
    print("Aún no lo resuelves.")
elif segundos_sin_cudnn > 0:
    print("✅ Registrado: compáralo con resultados['GRU']['segundos']")
else:
    print("❌ Revisa tu respuesta")

## Resumen

- `Embedding` + una capa recurrente leen el texto en orden.
- LSTM y GRU recuerdan más lejos que SimpleRNN; la bidireccional usa el contexto de ambos lados.
- En GPU, Keras acelera LSTM y GRU con **cuDNN** si se usan sus opciones por defecto.

**Fuentes:** [Keras · LSTM (condiciones para cuDNN)](https://keras.io/api/layers/recurrent_layers/lstm/) · [Keras · GRU](https://keras.io/api/layers/recurrent_layers/gru/) · [Keras · IMDB](https://keras.io/api/datasets/imdb/) · [Maas et al. (2011), críticas de IMDB](https://ai.stanford.edu/~amaas/data/sentiment/)